[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Saheb/rl-notebooks/blob/main/exercises/17_hidden_board.ipynb)

# 🃏 Notebook 17 — The Hidden Board
### Counterfactual Regret Minimization (CFR) & Poker

**Series**: RL Notebook Series · Act VI — Grandmasters · Post 17 of 17

---

## The Blind Spot

In Notebook 16, AlphaZero conquered Tic-Tac-Toe (and by extension, chess and Go).
But all these games share one property: **perfect information** — both players see the entire board.

**Poker** breaks this completely:
- You can see your own cards but **not your opponent's**
- The optimal strategy is **mixed** (probabilistic) — you *must* randomize
- **Bluffing** is not a psychological trick but a mathematical necessity

### Why MCTS Fails

MCTS works by simulating future game states. But in poker, you don't know what cards
your opponent holds — you can't simulate a state you can't see. We need a fundamentally
different approach.

### The Solution: CFR

**Counterfactual Regret Minimization** (Zinkevich et al., 2007) iteratively improves
a strategy by minimizing *regret* — how much the player wishes they had played differently.
The remarkable result: the **average strategy** converges to a **Nash equilibrium**,
a strategy that **cannot be exploited** by any opponent.

Today we'll build CFR from scratch on **Kuhn Poker** — the simplest non-trivial poker game —
and watch **bluffing emerge naturally from the math**.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from collections import defaultdict
import random

%matplotlib inline
plt.rcParams.update({'figure.figsize': (10, 5), 'axes.grid': True, 'grid.alpha': 0.3})
np.random.seed(42)
random.seed(42)

## 1. Kuhn Poker — The Simplest Poker Game

**Kuhn Poker** (Harold Kuhn, 1950) is a toy poker game with only 3 cards and 2 players.
Despite its simplicity, it captures the essential challenge of imperfect information:
bluffing, calling, and folding all emerge naturally.

### Rules
- **Deck**: just 3 cards — Jack (J), Queen (Q), King (K)
- **Deal**: each player gets 1 card. The third card is unseen.
- **Ante**: both players put 1 chip in the pot (starting pot = 2)
- **Actions**: `bet` (add 1 chip) or `check`/`call`/`fold`

### Game Flow
```
Player 1 acts first:
  ├── check → Player 2 acts:
  │     ├── check → Showdown (higher card wins the pot)
  │     └── bet   → Player 1 acts:
  │           ├── call → Showdown (pot = 4)
  │           └── fold → Player 2 wins (pot = 3)
  └── bet   → Player 2 acts:
        ├── call → Showdown (pot = 4)
        └── fold → Player 1 wins (pot = 3)
```

### Card Rankings
K > Q > J — the higher card wins at showdown.

In [ ]:
CARDS = ['J', 'Q', 'K']
CARD_RANK = {'J': 0, 'Q': 1, 'K': 2}


class KuhnPoker:
    """Kuhn Poker game state.
    
    Information set string format: "card:history"
    Example: "Q:cb" means "I hold Queen, action history is check-then-bet"
    """
    def __init__(self):
        self.cards = None
        self.history = ''
        self.current_player = 0
    
    def deal(self):
        """Shuffle and deal cards to both players."""
        deck = CARDS.copy()
        random.shuffle(deck)
        self.cards = [deck[0], deck[1]]
        self.history = ''
        self.current_player = 0
        return self
    
    def get_info_set(self):
        """Return the information set string for the current player."""
        return f"{self.cards[self.current_player]}:{self.history}"
    
    def get_valid_actions(self):
        """Return valid actions for the current player."""
        if self.history == '':
            return ['b', 'c']
        elif self.history == 'c':
            return ['b', 'c']
        elif self.history == 'b':
            return ['k', 'f']
        elif self.history == 'cb':
            return ['k', 'f']
        return []
    
    def is_terminal(self):
        """Check if the game is over."""
        return self.history in ['cc', 'bc', 'bf', 'bk', 'cbk', 'cbf']
    
    def get_payoff(self):
        """Return payoff for Player 1 (zero-sum: Player 2 gets the negative)."""
        assert self.is_terminal()
        if self.history == 'bf':
            return +1
        if self.history == 'cbf':
            return -1
        if self.history == 'bc':
            return +1
        
        p1_rank = CARD_RANK[self.cards[0]]
        p2_rank = CARD_RANK[self.cards[1]]
        winner = 1 if p1_rank > p2_rank else -1
        
        if self.history in ['bk', 'cbk']:
            return winner * 2
        else:
            return winner * 1
    
    def play(self, action):
        """Apply an action and return a new game state."""
        new = KuhnPoker()
        new.cards = self.cards
        new.history = self.history + action
        if not new.is_terminal():
            if len(new.history) % 2 == 0:
                new.current_player = 0
            else:
                new.current_player = 1
            if new.history == 'cb':
                new.current_player = 0
        return new


# Quick test
def play_random_kuhn(verbose=False):
    game = KuhnPoker().deal()
    if verbose:
        print(f"Deal: P1={game.cards[0]}, P2={game.cards[1]}")
    while not game.is_terminal():
        actions = game.get_valid_actions()
        action = random.choice(actions)
        if verbose:
            info = game.get_info_set()
            print(f"  P{game.current_player+1} ({info}) plays '{action}'")
        game = game.play(action)
    payoff = game.get_payoff()
    if verbose:
        print(f"  Result: P1 {'wins' if payoff > 0 else 'loses'} {abs(payoff)} chip(s)")
    return payoff

print("5 random Kuhn Poker games:\n")
for i in range(5):
    print(f"Game {i+1}:")
    play_random_kuhn(verbose=True)
    print()

payoffs = [play_random_kuhn() for _ in range(10000)]
print(f"Average P1 payoff over 10,000 random games: {np.mean(payoffs):.4f}")

## 2. Information Sets — What You Know vs What Is

In **perfect-information** games (chess, Tic-Tac-Toe), each game state is unique and fully visible.
In **imperfect-information** games, multiple states look identical to a player — these form an
**information set**.

Example: You hold a Queen and the history is "check, bet" (info set `Q:cb`).
Your opponent could hold J or K — you don't know which. Both possibilities
are in the same information set. Your strategy must be the same for both.

In [ ]:
# Enumerate all information sets in Kuhn Poker
def enumerate_info_sets():
    info_sets = {}
    def walk(game):
        if game.is_terminal():
            return
        info = game.get_info_set()
        actions = game.get_valid_actions()
        if info not in info_sets:
            info_sets[info] = {'player': game.current_player, 'actions': actions}
        for action in actions:
            walk(game.play(action))
    for c1 in CARDS:
        for c2 in CARDS:
            if c1 != c2:
                game = KuhnPoker()
                game.cards = [c1, c2]
                game.history = ''
                game.current_player = 0
                walk(game)
    return info_sets

info_sets = enumerate_info_sets()

print(f"Kuhn Poker has {len(info_sets)} information sets:\n")
print(f"{'Info Set':<10} {'Player':<8} {'Actions':<15}")
print("─" * 35)
for info, data in sorted(info_sets.items()):
    action_str = ', '.join(data['actions'])
    print(f"{info:<10} P{data['player']+1:<7} {action_str:<15}")

## 3. Regret and Regret Matching

The key idea behind CFR is **regret**: after each round, we ask
*"how much do I wish I had played a different action?"*

### Regret Matching

**Regret matching** converts cumulative regrets into a strategy:

$$\sigma^{T+1}(I, a) = \begin{cases} \frac{\max(R^T(I,a), 0)}{\sum_{a'} \max(R^T(I,a'), 0)} & \text{if denominator} > 0 \\ \frac{1}{|A(I)|} & \text{otherwise (uniform)} \end{cases}$$

**Intuition**: play actions you **regret not having played** more often.
If all regrets are non-positive (you're happy with your choices), play uniformly.

In [ ]:
def regret_matching(regret_sum, actions):
    """Convert cumulative regrets into a strategy via regret matching.
    
    Args:
        regret_sum: dict {action: cumulative_regret}
        actions: list of valid actions
    
    Returns:
        strategy: dict {action: probability}
    """
    # ============================================================
    # TODO: Implement regret matching.
    #
    # 1. For each action, clip the regret to max(regret, 0)
    # 2. Sum the positive regrets
    # 3. If the sum > 0, normalize to get probabilities
    # 4. Otherwise, return uniform distribution: 1/len(actions) each
    # ============================================================
    raise NotImplementedError("Implement regret matching!")


# Quick test
print("Regret matching examples:")

r1 = {'b': 3.0, 'c': -1.0}
s1 = regret_matching(r1, ['b', 'c'])
print(f"  Regrets {r1} → Strategy {s1}")

r2 = {'b': 2.0, 'c': 2.0}
s2 = regret_matching(r2, ['b', 'c'])
print(f"  Regrets {r2} → Strategy {s2}")

r3 = {'b': -1.0, 'c': -2.0}
s3 = regret_matching(r3, ['b', 'c'])
print(f"  Regrets {r3} → Strategy {s3}  (all negative → uniform)")

<details>
<summary>🔍 <b>Hint: Regret Matching</b> (click to reveal)</summary>

```python
positive_regrets = {a: max(regret_sum.get(a, 0), 0) for a in actions}
total = sum(positive_regrets.values())
if total > 0:
    return {a: positive_regrets[a] / total for a in actions}
else:
    return {a: 1.0 / len(actions) for a in actions}
```

</details>

<details>
<summary>✅ <b>Checkpoint — expected values</b> (click to reveal)</summary>

- `regret_matching({'b': 3.0, 'c': -1.0}, ['b', 'c'])` → `{'b': 1.0, 'c': 0.0}`
- `regret_matching({'b': 2.0, 'c': 2.0}, ['b', 'c'])` → `{'b': 0.5, 'c': 0.5}`
- `regret_matching({'b': -1.0, 'c': -2.0}, ['b', 'c'])` → `{'b': 0.5, 'c': 0.5}`

</details>

## 4. Rock-Paper-Scissors Warmup

Before tackling CFR on a game tree, let's see regret matching in action on the simplest
possible game: **Rock-Paper-Scissors** (RPS).

In [ ]:
RPS_ACTIONS = ['R', 'P', 'S']
RPS_PAYOFF = {
    ('R', 'R'): 0, ('R', 'P'): -1, ('R', 'S'): +1,
    ('P', 'R'): +1, ('P', 'P'): 0, ('P', 'S'): -1,
    ('S', 'R'): -1, ('S', 'P'): +1, ('S', 'S'): 0,
}


def train_rps(opponent_strategy, num_iterations=1000):
    """Train a regret-matching player against a fixed opponent strategy.
    
    Args:
        opponent_strategy: dict {'R': prob, 'P': prob, 'S': prob}
    
    Returns:
        average_strategy, strategy_history
    """
    regret_sum = {a: 0.0 for a in RPS_ACTIONS}
    strategy_sum = {a: 0.0 for a in RPS_ACTIONS}
    strategy_history = []
    
    for t in range(num_iterations):
        # Get current strategy from regret matching
        strategy = regret_matching(regret_sum, RPS_ACTIONS)
        
        # Sample actions
        my_action = np.random.choice(RPS_ACTIONS, p=[strategy[a] for a in RPS_ACTIONS])
        opp_action = np.random.choice(RPS_ACTIONS, p=[opponent_strategy[a] for a in RPS_ACTIONS])
        
        # ============================================================
        # TODO: Compute regrets and accumulate them.
        #
        # For each action I *could* have played:
        #   regret = payoff(that_action, opp_action) - payoff(my_action, opp_action)
        # Add each regret to regret_sum[action].
        #
        # Also accumulate the current strategy into strategy_sum for averaging.
        # ============================================================
        raise NotImplementedError("Compute and accumulate regrets!")
        
        strategy_history.append(strategy.copy())
    
    total = sum(strategy_sum.values())
    average_strategy = {a: strategy_sum[a] / total for a in RPS_ACTIONS}
    return average_strategy, strategy_history


# Test against biased opponent
biased_opp = {'R': 0.4, 'P': 0.3, 'S': 0.3}
avg_strat, hist = train_rps(biased_opp, 5000)
print(f"vs Biased (40R/30P/30S):")
print(f"  Learned: R={avg_strat['R']:.3f}  P={avg_strat['P']:.3f}  S={avg_strat['S']:.3f}")
print(f"  Expected: mostly Paper (beats Rock)\n")

# Test against uniform
uniform_opp = {'R': 1/3, 'P': 1/3, 'S': 1/3}
avg_strat2, hist2 = train_rps(uniform_opp, 5000)
print(f"vs Uniform (33/33/33):")
print(f"  Learned: R={avg_strat2['R']:.3f}  P={avg_strat2['P']:.3f}  S={avg_strat2['S']:.3f}")
print(f"  Expected: ~1/3 each (Nash equilibrium)")

<details>
<summary>🔍 <b>Hint: RPS Regrets</b> (click to reveal)</summary>

```python
my_payoff = RPS_PAYOFF[(my_action, opp_action)]
for a in RPS_ACTIONS:
    counterfactual_payoff = RPS_PAYOFF[(a, opp_action)]
    regret_sum[a] += counterfactual_payoff - my_payoff

for a in RPS_ACTIONS:
    strategy_sum[a] += strategy[a]
```

The regret for action `a` is: "what I would have gotten" minus "what I actually got".

</details>

## 5. Counterfactual Regret Minimization (CFR)

Now we scale regret matching from a one-shot game (RPS) to a sequential game tree (Kuhn Poker).

CFR traverses the game tree recursively. At each information set:
1. Get the current strategy from regret matching
2. Recurse for each action to compute the expected value of that action
3. Compute the counterfactual regret: value(action) - value(strategy)
4. Weight by the **opponent's reach probability** and accumulate

The **average strategy** across all iterations converges to Nash equilibrium.

In [ ]:
class CFRTrainer:
    """Counterfactual Regret Minimization for Kuhn Poker."""
    
    def __init__(self):
        self.regret_sum = defaultdict(lambda: defaultdict(float))
        self.strategy_sum = defaultdict(lambda: defaultdict(float))
        self.info_set_actions = {}
    
    def get_strategy(self, info_set, actions):
        """Get current strategy via regret matching."""
        self.info_set_actions[info_set] = actions
        return regret_matching(self.regret_sum[info_set], actions)
    
    def cfr(self, game, reach_0, reach_1):
        """Recursive CFR traversal. Returns expected utility for Player 1.
        
        Args:
            game: current KuhnPoker state
            reach_0: probability that Player 0's strategy reaches this node
            reach_1: probability that Player 1's strategy reaches this node
        """
        # Terminal node: return payoff
        if game.is_terminal():
            return game.get_payoff()
        
        player = game.current_player
        info_set = game.get_info_set()
        actions = game.get_valid_actions()
        
        # Get current strategy from regret matching
        strategy = self.get_strategy(info_set, actions)
        
        # ============================================================
        # TODO: Implement the CFR recursion.
        #
        # 1. For each action, recurse to get its value (action_values[action]).
        #    When recursing, multiply the current player's reach probability
        #    by strategy[action].
        #
        # 2. Compute node_value = sum of strategy[a] * action_values[a]
        #
        # 3. Accumulate counterfactual regrets:
        #    For each action, regret = action_values[action] - node_value
        #    Weighted by the OPPONENT's reach probability.
        #    For P0: self.regret_sum[info_set][a] += reach_1 * regret
        #    For P1: self.regret_sum[info_set][a] += reach_0 * (-regret)
        #    (negated because action_values are from P1's perspective)
        #
        # 4. Accumulate strategy_sum for averaging:
        #    self.strategy_sum[info_set][a] += my_reach * strategy[a]
        #
        # 5. Return node_value
        # ============================================================
        raise NotImplementedError("Implement CFR!")
    
    def train(self, num_iterations=10000):
        """Run CFR for num_iterations."""
        exploitability_history = []
        for t in range(num_iterations):
            for c1 in CARDS:
                for c2 in CARDS:
                    if c1 != c2:
                        game = KuhnPoker()
                        game.cards = [c1, c2]
                        game.history = ''
                        game.current_player = 0
                        self.cfr(game, 1.0, 1.0)
            if (t + 1) % 100 == 0:
                exploit = self.compute_exploitability()
                exploitability_history.append((t + 1, exploit))
        return exploitability_history
    
    def get_average_strategy(self):
        """Compute the average strategy across all iterations."""
        # ============================================================
        # TODO: Normalize strategy_sum to get the average strategy.
        #
        # For each info set, sum up strategy_sum[info_set][a] for all actions.
        # If total > 0, divide each action's sum by the total.
        # Otherwise, return uniform.
        # ============================================================
        raise NotImplementedError("Compute average strategy!")
    
    def compute_exploitability(self):
        """Compute how exploitable the current average strategy is."""
        avg_strategy = self.get_average_strategy()
        br_value_0 = self._best_response_value(avg_strategy, 0)
        br_value_1 = self._best_response_value(avg_strategy, 1)
        return br_value_0 + br_value_1
    
    def _best_response_value(self, strategy, br_player):
        total = 0.0
        count = 0
        for c1 in CARDS:
            for c2 in CARDS:
                if c1 != c2:
                    game = KuhnPoker()
                    game.cards = [c1, c2]
                    game.history = ''
                    game.current_player = 0
                    total += self._br_traverse(game, strategy, br_player)
                    count += 1
        return total / count
    
    def _br_traverse(self, game, strategy, br_player):
        if game.is_terminal():
            payoff = game.get_payoff()
            return payoff if br_player == 0 else -payoff
        player = game.current_player
        info_set = game.get_info_set()
        actions = game.get_valid_actions()
        if player == br_player:
            values = [self._br_traverse(game.play(a), strategy, br_player) for a in actions]
            return max(values)
        else:
            strat = strategy.get(info_set, {a: 1.0/len(actions) for a in actions})
            return sum(strat[a] * self._br_traverse(game.play(a), strategy, br_player) for a in actions)

<details>
<summary>🔍 <b>Hint: CFR Recursion</b> (click to reveal)</summary>

```python
action_values = {}
node_value = 0.0

for action in actions:
    next_game = game.play(action)
    if player == 0:
        action_values[action] = self.cfr(next_game, reach_0 * strategy[action], reach_1)
    else:
        action_values[action] = self.cfr(next_game, reach_0, reach_1 * strategy[action])
    node_value += strategy[action] * action_values[action]

opponent_reach = reach_1 if player == 0 else reach_0
for action in actions:
    regret = action_values[action] - node_value
    if player == 0:
        self.regret_sum[info_set][action] += opponent_reach * regret
    else:
        self.regret_sum[info_set][action] += opponent_reach * (-regret)

my_reach = reach_0 if player == 0 else reach_1
for action in actions:
    self.strategy_sum[info_set][action] += my_reach * strategy[action]

return node_value
```

</details>

<details>
<summary>🔍 <b>Hint: Average Strategy</b> (click to reveal)</summary>

```python
def get_average_strategy(self):
    avg = {}
    for info_set, action_sums in self.strategy_sum.items():
        actions = self.info_set_actions[info_set]
        total = sum(action_sums[a] for a in actions)
        if total > 0:
            avg[info_set] = {a: action_sums[a] / total for a in actions}
        else:
            avg[info_set] = {a: 1.0 / len(actions) for a in actions}
    return avg
```

</details>

## 6. Training CFR on Kuhn Poker

In [ ]:
print("Training CFR on Kuhn Poker (10,000 iterations)...\n")
cfr = CFRTrainer()
exploit_history = cfr.train(num_iterations=10000)

# Plot exploitability convergence
iters, exploits = zip(*exploit_history)
fig, ax = plt.subplots(figsize=(10, 4))
ax.plot(iters, exploits, color='#6366f1', linewidth=2)
ax.set_xlabel('CFR Iteration')
ax.set_ylabel('Exploitability')
ax.set_title('CFR Convergence on Kuhn Poker')
ax.axhline(y=0, color='#22c55e', linestyle='--', alpha=0.5, label='Nash (0)')
ax.legend()
plt.tight_layout()
plt.show()

print(f"Final exploitability: {exploits[-1]:.6f}")

In [ ]:
# Print the converged strategy
avg_strategy = cfr.get_average_strategy()

print("Converged Average Strategy (Nash Equilibrium Approximation):\n")
print(f"{'Info Set':<10} {'Player':<8} {'Action 1':<20} {'Action 2':<20}")
print("─" * 60)

for info_set in sorted(avg_strategy.keys()):
    actions = cfr.info_set_actions[info_set]
    player = info_sets[info_set]['player']
    strat = avg_strategy[info_set]
    
    parts = []
    for a in actions:
        label = {'b': 'bet', 'c': 'check', 'k': 'call', 'f': 'fold'}[a]
        parts.append(f"{label}: {strat[a]:.3f}")
    
    print(f"{info_set:<10} P{player+1:<7} {parts[0]:<20} {parts[1]:<20}")

## 7. Evaluating the Nash Equilibrium

Let's verify that our converged strategy is truly a Nash equilibrium by playing it
against various opponents.

In [ ]:
def evaluate_strategy(strategy, opponent_strategy, num_games=10000):
    """Play Kuhn Poker games and return average payoff for P1.
    
    strategy: dict {info_set: {action: prob}} for Player 1
    opponent_strategy: dict {info_set: {action: prob}} for Player 2
    """
    total_payoff = 0.0
    
    for _ in range(num_games):
        game = KuhnPoker().deal()
        
        while not game.is_terminal():
            info_set = game.get_info_set()
            actions = game.get_valid_actions()
            
            # ============================================================
            # TODO: Look up the strategy for the current player and
            # sample an action according to the probabilities.
            #
            # If player == 0, use `strategy`; if player == 1, use `opponent_strategy`.
            # If the info set is not found, use uniform random.
            # ============================================================
            raise NotImplementedError("Sample action from strategy!")
            
            game = game.play(action)
        
        total_payoff += game.get_payoff()
    
    return total_payoff / num_games


# Test: Nash vs Nash → game value ≈ -1/18 ≈ -0.056
nash_value = evaluate_strategy(avg_strategy, avg_strategy, 50000)
print(f"Nash vs Nash:    avg P1 payoff = {nash_value:.4f}  (expected: -0.056)")

nash_vs_random = evaluate_strategy(avg_strategy, {}, 50000)
print(f"Nash vs Random:  avg P1 payoff = {nash_vs_random:.4f}  (should be positive)")

random_vs_nash = evaluate_strategy({}, avg_strategy, 50000)
print(f"Random vs Nash:  avg P1 payoff = {random_vs_nash:.4f}  (should be negative)")

<details>
<summary>🔍 <b>Hint: Evaluate Strategy</b> (click to reveal)</summary>

```python
if game.current_player == 0:
    strat = strategy.get(info_set, {a: 1/len(actions) for a in actions})
else:
    strat = opponent_strategy.get(info_set, {a: 1/len(actions) for a in actions})

probs = [strat.get(a, 1/len(actions)) for a in actions]
probs = [p / sum(probs) for p in probs]  # normalize
action = np.random.choice(actions, p=probs)
```

</details>

## 8. Bluffing Emerges from Math

The most remarkable result: **bluffing is not a human psychological trick — it emerges mathematically
from the Nash equilibrium.**

With a Jack (worst hand), the optimal strategy sometimes **bets** (bluffs).
Why? If you *never* bluff, your opponent knows a bet always means a strong hand and always folds.
But then bluffing becomes profitable — so you must bluff sometimes.

In [ ]:
# Visualize bluffing frequencies
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

p1_info_sets = ['J:', 'Q:', 'K:']
p1_bet_probs = [avg_strategy.get(info, {}).get('b', 0) for info in p1_info_sets]

ax = axes[0]
colors = ['#ef4444', '#f59e0b', '#22c55e']
bars = ax.bar(['Jack', 'Queen', 'King'], p1_bet_probs, color=colors, width=0.5)
ax.set_title('Player 1: Opening Bet Probability')
ax.set_ylabel('P(bet)')
ax.set_ylim(0, 1.1)
for bar, prob in zip(bars, p1_bet_probs):
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.03,
            f'{prob:.2f}', ha='center', fontsize=12, fontweight='bold')

p2_info_sets = ['J:b', 'Q:b', 'K:b']
p2_call_probs = [avg_strategy.get(info, {}).get('k', 0) for info in p2_info_sets]

ax = axes[1]
bars = ax.bar(['Jack', 'Queen', 'King'], p2_call_probs, color=colors, width=0.5)
ax.set_title('Player 2: Call Probability (after P1 bets)')
ax.set_ylabel('P(call)')
ax.set_ylim(0, 1.1)
for bar, prob in zip(bars, p2_call_probs):
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.03,
            f'{prob:.2f}', ha='center', fontsize=12, fontweight='bold')

plt.tight_layout()
plt.show()

print("Key observations:")
print("  - Jack (worst hand): P1 sometimes BLUFFS (bets ~1/3 of the time)")
print("  - Queen (middle):    P1 usually checks, P2 calls ~1/3 of bets")  
print("  - King (best hand):  P1 often bets, P2 always calls")
print("\n  Bluffing is mathematically optimal — not a psychological trick!")

## 9. Scaling Up — From Kuhn to Texas Hold'em

Kuhn Poker has just **12 information sets**. Texas Hold'em has approximately **10^161**.

| System | Year | Key Innovation |
| :--- | :---: | :--- |
| **CFR** | 2007 | Base algorithm (what we built today) |
| **Monte Carlo CFR** | 2009 | Sample the tree instead of full traversal |
| **Abstraction** | 2015 | Group similar hands together (reduce info sets) |
| **Libratus** | 2017 | Beat top humans at heads-up no-limit Hold'em |
| **Deep CFR** | 2019 | Use neural nets to approximate the strategy |
| **Pluribus** | 2019 | Beat 6 human professionals simultaneously |

## 10. Recap — The Hidden Board

### What We Learned

| Concept | Key Idea |
| :--- | :--- |
| **Information sets** | Groups of states indistinguishable to a player |
| **Nash equilibrium** | Strategy no player can improve by deviating |
| **Regret** | How much you wish you had played differently |
| **Regret matching** | Convert cumulative regrets → strategy probabilities |
| **CFR** | Iterate regret matching across the game tree |
| **Average strategy** | Converges to Nash at $O(1/\sqrt{T})$ |
| **Bluffing** | Emerges mathematically — not psychological |

### Key Equations

**Regret matching:**
$$\sigma^{T+1}(I, a) = \frac{\max(R^T(I,a), 0)}{\sum_{a'} \max(R^T(I,a'), 0)}$$

**Counterfactual value:**
$$v_i(I) = \sum_{a \in A(I)} \sigma(I, a) \cdot v_i(I \to a)$$

**Regret update (Player 0 at info set $I$):**
$$R^{T+1}(I, a) = R^T(I, a) + \text{reach}_1 \cdot [v_i(I \to a) - v_i(I)]$$

**Convergence:**
$$\text{exploitability} \to 0 \quad \text{at} \quad O\!\left(\frac{1}{\sqrt{T}}\right)$$

---

### The Full Journey

| Act | Notebook | Algorithm | Key Idea |
| :---: | :--- | :--- | :--- |
| I | 01 Gridworld | Dynamic Programming | Bellman equations, value iteration |
| I | 02 Inventory | Stochastic DP | Uncertainty in transitions |
| II | 03 Bandit | $\varepsilon$-greedy / UCB | Exploration vs exploitation |
| II | 04 Contextual | LinUCB | Context-aware bandits |
| III | 05 Monte Carlo | MC Control | Learning from complete episodes |
| III | 06 TD Learning | Q-Learning / SARSA | Bootstrapping, TD(0) |
| IV | 07 DQN | Deep Q-Network | Neural function approximation |
| IV | 08 Double DQN | Double DQN | Overestimation fix |
| V | 09 REINFORCE | Policy Gradient | Directly optimize the policy |
| V | 10 Actor-Critic | A2C | Value function as baseline |
| V | 11 PPO | Proximal Policy | Stable policy updates with clipping |
| V | 12 GAE | Generalized Advantage | Variance reduction in PG |
| VI | 13 Dream Machine | Model-Based RL | Learn a world model, plan inside it |
| VI | 14 GRPO | Group Relative PO | RLHF without a value network |
| VI | 15 DPO | Direct Preference Opt | Alignment without RL at all |
| **VI** | **16 Infinite Curriculum** | **AlphaZero** | **Self-play + MCTS + neural nets** |
| **VI** | **17 Hidden Board** | **CFR** | **Nash equilibrium from regret** |

**You've completed the full RL Notebook Series.** From tabular MDPs to bluffing poker agents — the core ideas of modern reinforcement learning are now in your toolkit.